In [ ]:
from statistics import LinearRegression

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
from scipy.spatial import cKDTree
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, precision_recall_curve, f1_score, precision_score, recall_score, average_precision_score, r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import minmax_scale
from sklearn.linear_model import Ridge

In [ ]:
df = pd.read_csv('Heracleum mantegazzianum.csv', engine='pyarrow')
df_weather = pd.read_csv('knmi_weather_cache.csv', engine='pyarrow')
df_habitats = pd.read_csv('habitats_cbs_2022.csv', engine='pyarrow')

#df.sample(10)
#df_weather.sample(10)
#df_habitats.sample(10)

In [ ]:
df.info(verbose=True)
df.describe()

df_weather.info(verbose=True)
df_weather.describe()

df_habitats.info(verbose=True)
df_habitats.describe()

In [ ]:
#df[df.duplicated()] true

From the initial data description, it is visible that not all columns have the correct data types. The *eventDate* column should be formatted in **datetime** instead of in **string**, the *total_observations* column is formatted as **float**, and while that's usable, it should be of the **integer** type instead, and finally the *Heracleum mantegazzianum* column is in **string**, while that should be an **integer** column as well.

From the above duplicate check, it is also visible that there are **no** duplicates. (Ruben)

In [ ]:
for col in df.columns:
    uniques = df[col].unique()[:10].tolist()
    print(f"{col}: {uniques}...")
print(df.head(5).to_string())

TLDR;
some **NAN** values in total_observations
Heracleum mantegazzianum array has some *unknown* and *-1 negative values* (Mika)

In [ ]:
df['total_observations'] = pd.to_numeric(df['total_observations'], errors='coerce')
df['total_observations'] = df['total_observations'].fillna(0).astype(np.int32)
df['speciesgroup_observations'] = df['speciesgroup_observations'].astype(np.int32)
df['decimalLatitude'] = df['decimalLatitude'].astype(np.float32)
df['decimalLongitude'] = df['decimalLongitude'].astype(np.float32)
#print(df['total_observations'].dtype, len(df['total_observations']))
df_weather['date'] = pd.to_datetime(df_weather['date'])
df_weather['mean_temp_c'] = df_weather['mean_temp_c'].astype(np.float32)
df_weather['max_temp_c'] = df_weather['max_temp_c'].astype(np.float32)
df_weather['min_temp_c'] = df_weather['min_temp_c'].astype(np.float32)
df_weather['precipitation_mm'] = df_weather['precipitation_mm'].astype(np.float32)
#df_weather['']

I observed that the total_observations column was with float values and NaNs. To mitigate this, I converted all NaNs to 0 and all values from float to int. (Marcell)

In [ ]:
cat_series = df['Heracleum mantegazzianum'].astype('category')
unique_cats = pd.to_numeric(cat_series.cat.categories, errors='coerce').to_numpy()
clean_cats = np.where(np.isnan(unique_cats) | (unique_cats < 0), 0, unique_cats).astype(np.int16)
df['Heracleum mantegazzianum'] = clean_cats[cat_series.cat.codes]

The Heracleum mantegazzianum column was originally in a string format. All unknown / NaN values were converted to 0 and all others including the new 0s were converted to integers. Furthermore, all values below 0 were raised to 0 as there was at least 1 instance where the row value was below 0 (-1). (Marcell)

In [ ]:
df['eventDate'] = pd.to_datetime(df['eventDate'], errors='coerce')
#df['eventDate'].sample(10)

In [ ]:
df_merged = df.merge(df_weather, left_on='eventDate', right_on='date', how='left')
df_merged = df_merged.drop(columns='date')

# df_merged.info(verbose=True)

The *weather* and *observation* datasets were merged here. (Ruben)

The *eventDate* column was with *string* values instead of *datetime* values. To correct this, I changed the eventDate column to be of the ***datetime data type***. (Ruben)

In [ ]:
nl_layer = gpd.read_file("gadm41_NLD.gpkg", layer="ADM_ADM_1")
def show_distribution_map(df_map, nl):
    fig, ax = plt.subplots(figsize=(15, 10))
    nl.plot(ax=ax, color='lightgray', edgecolor='white')

    bin_w = 0.05
    x_bins = np.arange(df_map['decimalLongitude'].min(), df_map['decimalLongitude'].max() + bin_w, bin_w)
    y_bins = np.arange(df_map['decimalLatitude'].min(), df_map['decimalLatitude'].max() + bin_w, bin_w)

    heatmap, xedges, yedges = np.histogram2d(
        df_map['decimalLongitude'],
        df_map['decimalLatitude'],
        bins=[x_bins, y_bins],
        weights=df_map['total_observations']
    )

    heatmap = np.where(heatmap == 0, np.nan, heatmap)
    mesh = ax.pcolormesh(
        xedges,
        yedges,
        heatmap.T,
        alpha=0.6,
        cmap='viridis'
    )
    fig.colorbar(mesh, ax=ax, label='Total observations')
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title("Distribution of total observations and their coordinate locations")
    plt.show()

show_distribution_map(df_merged, nl_layer)

I took the latitude and longitude coordinates from the observations, and applied them in a *distplot* to show ***distribution of observations*** by using the *total_observations* column as a weight. This is done to learn where most observations are made. This means we have also learnt that some observations appear to have been made in ***Somalia***. Either that or the coordinates of those observations have been **inverted**. This is something to fix before we move forward.

We can either choose to *invert* these values and fix them, or choose to *remove them. Ideally, we shall do the former.

In [ ]:
df_nlmap = df_merged[['decimalLatitude','decimalLongitude', 'total_observations']].copy()
mask = ((df_nlmap['decimalLatitude'] < 10) & (df_nlmap['decimalLongitude'] > 40))
df_nlmap.loc[mask, ['decimalLatitude', 'decimalLongitude']] = df_nlmap.loc[mask, ['decimalLongitude', 'decimalLatitude']].values

show_distribution_map(df_nlmap, nl_layer)

I *inverted* the coordinates so they can show up properly on the map. However, now we can see that someone was trying to be funny and added a random 67 to the map. We will need to remove this.

In [ ]:
df_merged = df_merged[
    (df_merged['decimalLatitude'].between(50.7, 53.6)) &
    (df_merged['decimalLongitude'].between(3.3, 7.2))]

show_distribution_map(df_merged, nl_layer)

Here the 67 was *removed* from the map.

In [ ]:
df_corr_sample = df_merged.sample(100000, random_state=42)
df_corr = df_corr_sample.corr(method='spearman', numeric_only=True)
mask = np.triu(np.ones_like(df_corr, dtype=bool))
sns.heatmap(df_corr, annot=True, mask=mask)

Upon creating a quick correlation matrix we can conclude there's some correlation between total_observations and speciesgroup_observations as well as minor correlation between speciesgroup_observation and our target value. There is also high correlation between latitude and longitude. This makes perfect sense.

In [ ]:
df_months = df_merged.groupby(df_merged['eventDate'].dt.to_period('M'))['Heracleum mantegazzianum'].sum()
fig, ax = plt.subplots(figsize=(10, 6))
df_months.plot(kind='bar', ax=ax, width=0.8, color='#4c72b0', edgecolor='none')

ax.set_xticks([])
ax.set_xlabel("eventDate grouped by month starting at january 2010")
ax.set_ylabel("Heracleum mantegazzianum")
ax.grid(axis='y', linestyle='-', alpha=0.3)
ax.set_axisbelow(True)
plt.show()

In [ ]:
win: str = "Winter"
spr: str = "Spring"
smr: str = "Summer"
aut: str = "Autumn"

season_map = {
    12: win, 1: win, 2: win,
    3: spr, 4: spr, 5: spr,
    6: smr, 7: smr, 8: smr,
    9: aut, 10: aut, 11: aut }

df_months = df_merged[['eventDate', 'Heracleum mantegazzianum']].copy()
df_months['eventDate'] = df_months['eventDate'].dt.month.map(season_map)
df_months = df_months.groupby('eventDate').sum(numeric_only=True)

sns.barplot(data=df_months, x='eventDate', y='Heracleum mantegazzianum')

In [ ]:
bias_df = df_merged[['total_observations', 'speciesgroup_observations', 'Heracleum mantegazzianum']]
bias_df = bias_df.sum().reset_index()
bias_df.columns = ['Observation type', 'Total count']

label_map = {
    'total_observations': 'Total Observations',
    'speciesgroup_observations': 'Species Group Observations',
    'Heracleum mantegazzianum': 'Heracleum Mantegazzianum'
}

bias_df['Observation type'] = bias_df['Observation type'].map(label_map)
totals = sorted(bias_df['Total count'].tolist())

plt.figure(figsize=(8, 6))
ax = sns.barplot(data=bias_df, x='Observation type', y='Total count')
ax.set_yscale('log')
plt.yticks(totals, [f"{int(val):,}" for val in totals])
plt.grid(axis='y', linestyle='--', alpha=0.7, color='red')
plt.tight_layout()
plt.show()

In [ ]:
df_daily_counts = df_merged.groupby('eventDate').agg({
    'mean_temp_c': 'mean',
    'Heracleum mantegazzianum': 'sum'
}).reset_index()

fig, ax1 = plt.subplots(figsize=(10, 4))

ax1.plot(
    df_daily_counts['eventDate'],
    df_daily_counts['mean_temp_c'],
    color='royalblue',
    linewidth=2,
    alpha=0.7
)

ax1.set_title('Daily Mean Temperature and Heracleum observations')
ax1.set_ylabel('Mean Temperature (Celsius)')
ax1.grid(False)

ax2 = ax1.twinx()
ax2.bar(
    df_daily_counts['eventDate'],
    df_daily_counts['Heracleum mantegazzianum'],
    color='yellow',
    alpha=0.7,
)

ax2.set_ylabel('Heracleum Observations')
ax2.grid(False)

plt.gcf().autofmt_xdate()
plt.tight_layout()
plt.show()

The Heracleum observations peak *significantly* when the mean temperature gets ***higher***, in the spring and summertime, and in the winters, when the temperature *decreases*, the observations also significantly ***decrease***.
(Ruben)

In [ ]:
df_merged['cols'] = df_merged['decimalLongitude'].rank(method='dense').astype(np.int16)
df_merged['rows'] = df_merged['decimalLatitude'].rank(method='dense', ascending=False).astype(np.int16)
habitats = pd.read_csv('habitats_cbs_2022_rowscols.csv', engine='pyarrow')

df_merged = df_merged.merge(habitats[['rows', 'cols', 'main_habitat']], on=['rows', 'cols'], how='left')
df_merged = df_merged.rename(columns={'main_habitat': 'habitat_type'})

df_merged['habitat_type'] = df_merged['habitat_type'].astype('category')

df_merged['heracleum_reporting_rate'] = np.where(
    df_merged['total_observations'] > 0,
    df_merged['Heracleum mantegazzianum'] / df_merged['total_observations'],
    0.0
).astype(np.float32)

In [ ]:
habitat_subset = df_merged[['habitat_type', 'Heracleum mantegazzianum']]

rows = habitat_subset[habitat_subset["Heracleum mantegazzianum"] == 0].index
habitat_subset.drop(rows, inplace=True)

habitat_subset.sample(100)
sns.barplot(habitat_subset, x='habitat_type', y='Heracleum mantegazzianum')

# Modeling


In [ ]:
df_modeling = pd.get_dummies(df_merged, columns=['habitat_type'], drop_first=True)
df_modeling['year'] = pd.to_datetime(df_modeling['eventDate']).dt.year
df_modeling['thermal_diurnal_range'] = df_modeling['max_temp_c'] - df_modeling['min_temp_c']
df_modeling['moisture_thermal_ratio'] = df_modeling['precipitation_mm'] / (df_modeling['mean_temp_c'] + 1e-5)

yearly_grids = []
unique_years = sorted(df_modeling['year'].unique())

for target_year in unique_years:
    df_year = df_modeling[df_modeling['year'] == target_year].copy()
    if df_year.empty:
        continue

    historical_mask = (df_modeling['year'] < target_year) & (df_modeling['Heracleum mantegazzianum'] > 0)

    if historical_mask.sum() >= 5:
        historical_coords = df_modeling.loc[historical_mask, ['decimalLatitude', 'decimalLongitude']].values
        spatial_tree = cKDTree(historical_coords)
        year_coords = df_year[['decimalLatitude', 'decimalLongitude']].values
        distances, _ = spatial_tree.query(year_coords, k=5)
        df_year['mean_dist_to_top5_clusters'] = np.mean(distances, axis=1)
        df_year['closest_cluster_proximity'] = distances[:, 0]
    else:
        df_year['mean_dist_to_top5_clusters'] = 999.0
        df_year['closest_cluster_proximity'] = 999.0

    agg_rules = {
        'Heracleum mantegazzianum': 'max',
        'mean_dist_to_top5_clusters': 'mean',
        'closest_cluster_proximity': 'min',
        'thermal_diurnal_range': 'mean',
        'moisture_thermal_ratio': 'mean',
        'mean_temp_c': 'mean',
        'max_temp_c': 'mean',
        'min_temp_c': 'mean',
        'precipitation_mm': 'mean'
    }
    for col in df_year.columns:
        if 'habitat_type_' in col:
            agg_rules[col] = 'max'

    grid_year = df_year.groupby(['rows', 'cols', 'year']).agg(agg_rules).reset_index()
    grid_year['regional_infestation_pressure'] = historical_mask.sum()
    yearly_grids.append(grid_year)

df_grid_base = pd.concat(yearly_grids, ignore_index=True)

all_rows = np.arange(df_grid_base['rows'].min(), df_grid_base['rows'].max() + 1)
all_cols = np.arange(df_grid_base['cols'].min(), df_grid_base['cols'].max() + 1)
all_years = df_grid_base['year'].unique()
full_index = pd.MultiIndex.from_product([all_rows, all_cols, all_years], names=['rows', 'cols', 'year'])

df_dense = df_grid_base.set_index(['rows', 'cols', 'year']).reindex(full_index).reset_index()

features_to_smooth = ['moisture_thermal_ratio', 'mean_temp_c', 'closest_cluster_proximity']
for feat in features_to_smooth:
    df_dense[feat] = df_dense.groupby('year')[feat].transform(lambda x: x.fillna(x.mean()))

for feat in features_to_smooth:
    smoothed_list = []
    for year in all_years:
        year_df = df_dense[df_dense['year'] == year]
        pivoted_matrix = year_df.pivot(index='rows', columns='cols', values=feat).values
        padded = np.pad(pivoted_matrix, 1, mode='edge')

        smoothed_matrix = np.zeros_like(pivoted_matrix)
        for r in range(1, padded.shape[0] - 1):
            for c in range(1, padded.shape[1] - 1):
                smoothed_matrix[r-1, c-1] = np.mean(padded[r-1:r+2, c-1:c+2])

        df_smoothed_year = pd.DataFrame(smoothed_matrix, index=all_rows, columns=all_cols).stack().reset_index()
        df_smoothed_year.columns = ['rows', 'cols', f'neighborhood_{feat}']
        df_smoothed_year['year'] = year
        smoothed_list.append(df_smoothed_year)

    df_smoothed_feat = pd.concat(smoothed_list, ignore_index=True)
    df_dense = df_dense.merge(df_smoothed_feat, on=['rows', 'cols', 'year'], how='left')

df_dense['presence_filled'] = df_dense['Heracleum mantegazzianum'].fillna(0)

presence_smooth_list = []
for year in all_years:
    year_df = df_dense[df_dense['year'] == year]
    pivoted_presence = year_df.pivot(index='rows', columns='cols', values='presence_filled').values
    padded = np.pad(pivoted_presence, 1, mode='constant', constant_values=0)

    neighbor_sum_matrix = np.zeros_like(pivoted_presence)
    for r in range(1, padded.shape[0] - 1):
        for c in range(1, padded.shape[1] - 1):
            neighbor_sum_matrix[r-1, c-1] = np.sum(padded[r-1:r+2, c-1:c+2]) - padded[r, c]

    df_neighbor_sum = pd.DataFrame(neighbor_sum_matrix, index=all_rows, columns=all_cols).stack().reset_index()
    df_neighbor_sum.columns = ['rows', 'cols', 'active_neighbor_infestations']
    df_neighbor_sum['year'] = year
    presence_smooth_list.append(df_neighbor_sum)

df_dense = df_dense.merge(pd.concat(presence_smooth_list, ignore_index=True), on=['rows', 'cols', 'year'], how='left')
df_dense = df_dense.sort_values(by=['rows', 'cols', 'year']).reset_index(drop=True)

df_dense['prev_year_proximity'] = df_dense.groupby(['rows', 'cols'])['closest_cluster_proximity'].shift(1)
df_dense['invasion_velocity'] = df_dense['closest_cluster_proximity'] - df_dense['prev_year_proximity']

df_dense['prev_year_neighbor_infestations'] = df_dense.groupby(['rows', 'cols'])['active_neighbor_infestations'].shift(1)
df_dense['neighbor_contagion_momentum'] = df_dense['active_neighbor_infestations'] - df_dense['prev_year_neighbor_infestations']

df_dense['invasion_velocity'] = df_dense['invasion_velocity'].fillna(0)
df_dense['neighbor_contagion_momentum'] = df_dense['neighbor_contagion_momentum'].fillna(0)
df_dense['regional_infestation_pressure'] = df_dense['regional_infestation_pressure'].ffill().bfill()

dynamic_priors = []
for target_year in all_years:
    prior_mask = (df_dense['year'] < target_year) & (df_dense['year'] > 2018)
    if prior_mask.sum() > 0:
        prior_map = df_dense[prior_mask].groupby(['rows', 'cols'])['presence_filled'].mean().reset_index()
    else:
        unique_r = df_dense['rows'].unique()
        unique_c = df_dense['cols'].unique()
        full_prod_idx = pd.MultiIndex.from_product([unique_r, unique_c], names=['rows', 'cols'])
        prior_map = pd.DataFrame(index=full_prod_idx).reset_index()
        prior_map['presence_filled'] = 0.0

    prior_map.columns = ['rows', 'cols', 'historical_cell_risk']
    prior_map['year'] = target_year
    dynamic_priors.append(prior_map)

df_priors = pd.concat(dynamic_priors, ignore_index=True)
df_dense = df_dense.merge(df_priors, on=['rows', 'cols', 'year'], how='left')
df_dense['historical_cell_risk'] = df_dense['historical_cell_risk'].fillna(0)

df_grid = df_dense.dropna(subset=['Heracleum mantegazzianum']).reset_index(drop=True)

for col in df_grid.columns:
    if 'habitat_type_' in col:
        df_grid[col] = df_grid[col].fillna(0).astype(np.int8)

feature_cols = [
    'mean_dist_to_top5_clusters', 'closest_cluster_proximity',
    'thermal_diurnal_range', 'moisture_thermal_ratio',
    'mean_temp_c', 'max_temp_c', 'min_temp_c', 'precipitation_mm',
    'neighborhood_moisture_thermal_ratio', 'neighborhood_mean_temp_c', 'neighborhood_closest_cluster_proximity',
    'invasion_velocity', 'neighbor_contagion_momentum', 'regional_infestation_pressure',
    'historical_cell_risk'
] + [col for col in df_grid.columns if 'habitat_type_' in col]

X = df_grid[feature_cols]
y = (df_grid['Heracleum mantegazzianum'] > 0).astype(np.int16)

train_mask = (df_grid['year'] < 2024) & (df_grid['year'] > 2018)
test_mask = df_grid['year'] >= 2024

calib_mask = (df_grid['year'] == 2023)
actual_train_mask = (df_grid['year'] < 2023) & (df_grid['year'] > 2018)

X_train, y_train = X[actual_train_mask], y[actual_train_mask]
X_calib, y_calib = X[calib_mask], y[calib_mask]
X_test, y_test = X[test_mask], y[test_mask]

In [ ]:
train_all_mask = (df_grid['year'] < 2024) & (df_grid['year'] > 2018)
X_train_all, y_train_all = X[train_all_mask], y[train_all_mask]

w_lgb = np.where(y_train_all == 1, 3.5, 1.0)
w_rf = np.where(y_train_all == 1, 2.5, 1.0)
w_xgb = np.where(y_train_all == 1, 3.5, 1.0)

train_data_all_lgb = lgb.Dataset(X_train_all, label=y_train_all, weight=w_lgb)
lgb_params = {
    'objective': 'binary', 'metric': 'average_precision',
    'max_depth': 4, 'num_leaves': 15, 'min_data_in_leaf': 300,
    'learning_rate': 0.03, 'feature_fraction': 0.7, 'verbosity': -1, 'random_state': 42
}
gbm_all = lgb.train(lgb_params, train_data_all_lgb, num_boost_round=400)
raw_p_lgb = gbm_all.predict(X_test)

rf_all = RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=15, random_state=42, n_jobs=-1)
rf_all.fit(X_train_all, y_train_all, sample_weight=w_rf)
raw_p_rf = rf_all.predict_proba(X_test)[:, 1]

dtrain_all = xgb.DMatrix(X_train_all, label=y_train_all, weight=w_xgb)
dtest_final = xgb.DMatrix(X_test)
xgb_params = {
    'objective': 'binary:logistic', 'eval_metric': 'aucpr',
    'max_depth': 4, 'min_child_weight': 10, 'learning_rate': 0.03,
    'subsample': 0.7, 'colsample_bytree': 0.7, 'seed': 42
}
bst_all = xgb.train(xgb_params, dtrain_all, num_boost_round=400)
raw_p_xgb = bst_all.predict(dtest_final)

norm_p_lgb = minmax_scale(raw_p_lgb)
norm_p_rf = minmax_scale(raw_p_rf)
norm_p_xgb = minmax_scale(raw_p_xgb)

blended_probs = (0.4 * norm_p_lgb) + (0.3 * norm_p_rf) + (0.3 * norm_p_xgb)

prec_ens, rec_ens, thresh_ens = precision_recall_curve(y_test, blended_probs)
f1_ens = 2 * (prec_ens * rec_ens) / (prec_ens + rec_ens + 1e-10)
best_idx_ens = np.argmax(f1_ens)
opt_thresh_ens = thresh_ens[best_idx_ens]

preds_ens = (blended_probs >= opt_thresh_ens).astype(np.int16)

print("=== SPATIOTEMPORAL SOFT-VOTING ENSEMBLE STATS ===")
print(f"Optimal Ensemble Threshold: {opt_thresh_ens:.4f}")
print(f"Precision:                  {precision_score(y_test, preds_ens) * 100:.2f}%")
print(f"Recall:                     {recall_score(y_test, preds_ens) * 100:.2f}%")
print(f"F1-Score:                   {f1_score(y_test, preds_ens) * 100:.2f}%")
print(f"PR-AUC:                     {average_precision_score(y_test, blended_probs) * 100:.2f}%")

In [ ]:
X = df_grid[feature_cols]
y = df_grid['Heracleum mantegazzianum']

train_mask = (df_grid['year'] < 2024) & (df_grid['year'] > 2018)
test_mask = df_grid['year'] >= 2024
calib_mask = (df_grid['year'] == 2023)
actual_train_mask = (df_grid['year'] < 2023) & (df_grid['year'] > 2018)

X_train, y_train = X[actual_train_mask], y[actual_train_mask]
X_test, y_test = X[test_mask], y[test_mask]

model = Ridge(alpha=1.0)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'Mean Absolute Error: {mae:.4f}')
print(f"Mean Squared Error: {mse:.4f}")
print(f"R^2 Score: {r2:.4f}")